In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import pytz
from zipfile import ZipFile

# Move the exported file to the destination folder
import os
import shutil  # Import shutil for moving files

In [2]:
local_tz = pytz.timezone('Asia/Jakarta')  # Replace with your time zone
now_local = datetime.now(local_tz)
date_format = "%d %B %Y" # Choose your desired format
date_str = now_local.strftime(date_format)

In [3]:
from google.colab import files
file = files.upload()

Saving Template List Delivery SLOT-SAMEDAY - SAMEDAY.csv to Template List Delivery SLOT-SAMEDAY - SAMEDAY.csv


In [4]:
# Define the directory where the uploaded file is expected to be
upload_dir = '/content/'

# Use a try-except block to handle potential errors during file processing
try:
    # Find all files in the upload directory that end with '.csv'
    csv_files = [file for file in os.listdir(upload_dir) if file.endswith('.csv')]

    # Check the number of CSV files found
    if len(csv_files) == 1:
        # If exactly one CSV file is found, get its filename and full path
        csv_filename = csv_files[0]
        file_path = os.path.join(upload_dir, csv_filename)

        # Print a message indicating the found CSV file
        print(f"Found CSV file: {csv_filename}")

        # Read the CSV file into a pandas DataFrame
        df = pd.read_csv(file_path)

        # Remove commas and convert 'total_price' and 'Zone_Price' to integers
        df['total_price'] = df['total_price'].astype(str).str.replace(',', '', regex=False).astype(int)
        df['Zone_Price'] = df['Zone_Price'].astype(str).str.replace(',', '', regex=False).astype(int)

        print("Successfully read the CSV file.")

    elif len(csv_files) == 0:
        # If no CSV files are found, print an error message
        print("Error: No CSV files were found in the directory.")

    else:
        # If multiple CSV files are found, print an error message and list the files
        print(f"Error: Multiple CSV files found. Please specify which one to use: {csv_files}")
        # You could add logic here to pick the newest file, or ask the user to choose.

except FileNotFoundError:
    # Handle the case where the upload directory does not exist
    print(f"Error: The directory '{upload_dir}' does not exist.")
except Exception as e:
    # Handle any other unexpected errors that might occur
    print(f"An unexpected error occurred: {e}")

Found CSV file: Template List Delivery SLOT-SAMEDAY - SAMEDAY.csv
Successfully read the CSV file.


In [5]:
df_columns = ['driver_name',    'district',     'customer_name',        'delivery_date',        'address',      'address_note', 'order_no',     'packaging_option',     'distance_in_km',       'hubs', 'total_price',  'external_note',        'internal_note',        'customer_note',        'time_slot',    'no_plastic',   'payment_method',       'latitude',     'longitude', 'shipping_number (Box #)']
df_backup = df.copy(deep=True)

df.loc[:, 'driver_name'] = df['driver_name'].str.upper()

# Sort by multiple columns:
df = df.sort_values(by=['hubs', 'driver_name', 'customer_name'], ascending=[True, True, True])

# Apply column filtering after sorting
df = df[df_columns]

df.head()

,driver_name,district,customer_name,delivery_date,address,address_note,order_no,packaging_option,distance_in_km,hubs,total_price,external_note,internal_note,customer_note,time_slot,no_plastic,payment_method,latitude,longitude,shipping_number (Box #)
1,DASH-KR-SETIYONO,kecamatan pulo gadung,Florencia,2026-03-14,"Jl. Duren I No.23, RT.15/RW.8, Rawamangun, Kec...",Jl duren 1 no 23,SA-HAM906MNNX2U-NR,PLASTIC,4.30,Kramat - JAKPUS,266100,NaN,NaN,NaN,slot-sameday,0,VA BCA,-6.20,106.89,229
3,DASH-KR-SETIYONO,kecamatan cempaka putih,Poppy Wijaya,2026-03-14,"Cempaka Putih Tengah, Cemp. Putih Tim., Kec. C...",Arandra Apartment - Tower Prosperity Unit P170...,SA-KW64O5GB5KKS-NR,PLASTIC,0.40,Kramat - JAKPUS,236400,NaN,NaN,NaN,slot-sameday,0,OVO Wallet,-6.18,106.86,251
4,DASH-KR-SETIYONO,kecamatan cempaka putih,Rishi,2026-03-14,"Cempaka putih tengah XXVI 26 A no,18 RT.4/RW.6...",Cempaka putih tengah 26a no 18.,SA-5OILY5VL5B1Y-NR,HALF MEDIUM BOX,0.80,Kramat - JAKPUS,166300,NaN,NaN,NaN,slot-sameday,0,VA BCA,-6.18,106.87,231
5,DASH-KR-SETIYONO,kecamatan matraman,Vania Utami,2026-03-14,"Jl. Galur Sari IX No.34, RT.12/RW.1, Utan Kayu...",Rumah samping Depan LaundryKu pagar hitam cat ...,SA-HP22P5GETZ3S-NR,NaN,2.81,Kramat - JAKPUS,156500,NaN,NaN,NaN,slot-sameday,0,BRI VA Transferpay!,-6.20,106.87,250
0,DASH-KR-SETIYONO,kecamatan pulo gadung,fadhil,2026-03-14,Jl. Jatinegara Kaum No.27,NaN,SA-KVYQQ53YO9UY-NR,PLASTIC,5.59,Kramat - JAKPUS,158000,NaN,NaN,NaN,slot-sameday,0,ShopeePay,-6.20,106.90,249


In [6]:
def cash_on_delivery_payment(row):
    payment_method_value = row.loc['payment_method']

    # Define a single background color based on conditions
    background_color = '#FFFFFF'  # Default color
    text_color = '#000000'  # Default text color

    if payment_method_value == 'Cash on Delivery':
        text_color = 'green'  # Green for Cash on Delivery

    # Return a list of styles for each cell in the row, including border
    return [
        f'background-color: {background_color}; color: {text_color}; border: 0.5px solid #D3D3D3' for _ in row
    ]

# Apply conditional formatting and borders
formatted_df = df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
        {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
    ])


# ...
formatted_df.to_excel("formated_df.xlsx", index=False)

In [7]:
# Extract data where driver_name has "BLITZ-" or "blitz-" (case-insensitive)
blitz_2_df = df[df['driver_name'].str.contains('BLITZ-|blitz-|3W|3w', case=True) & df['time_slot'].str.contains('slot-2|slot-sameday03', case=True)]

# Create an Excel writer object
filename = f"Blitz List Delivery {date_str} SLOT-2.xlsx"
# Export the filtered DataFrame to Excel
if not blitz_2_df.empty:
  formatted_df = blitz_2_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")


  destination_folder_slot_2 = "/content/exported_files/slot_2"  # Replace with your desired folder path
  os.makedirs(destination_folder_slot_2, exist_ok=True)
  shutil.move(filename, destination_folder_slot_2)

  print(f"File moved to: {os.path.join(destination_folder_slot_2, filename)}")
else:
  print(f"No data to export for {filename}")

No data to export for Blitz List Delivery 14 March 2026 SLOT-2.xlsx


In [8]:
# Extract data where driver_name has "BLITZ-" or "blitz-" (case-insensitive)
blitz_13_df = df[df['driver_name'].str.contains('BLITZ-|blitz-|3W|3w', case=True) & df['time_slot'].str.contains('slot-13|slot-sameday$', case=True)]


# writer = pd.ExcelWriter(f'Blitz List Delivery {date_str}.xlsx')
filename = f"Blitz List Delivery {date_str} SLOT-SAMEDAY.xlsx"
# Export the filtered DataFrame to Excel
if not blitz_13_df.empty:
  formatted_df = blitz_13_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  destination_folder_slot_13 = "/content/exported_files/slot_13"  # Replace with your desired folder path
  os.makedirs(destination_folder_slot_13, exist_ok=True)
  shutil.move(filename, destination_folder_slot_13)

  print(f"File moved to: {os.path.join(destination_folder_slot_13, filename)}")
else:
  print(f"No data to export for {filename}")

Blitz List Delivery 14 March 2026 SLOT-SAMEDAY.xlsx is exported succesfully
File moved to: /content/exported_files/slot_13/Blitz List Delivery 14 March 2026 SLOT-SAMEDAY.xlsx


In [9]:
koperasi_2_df = df[df['driver_name'].str.contains('KOP-|kop-', case=True) & df['time_slot'].str.contains('slot-2|slot-sameday03', case=True)]

# Create an Excel writer object
filename = f"Koperasi List Delivery {date_str} SLOT-2.xlsx"
# Export the filtered DataFrame to Excel
if not koperasi_2_df.empty:
  formatted_df = koperasi_2_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  os.makedirs(destination_folder_slot_2, exist_ok=True)
  shutil.move(filename, destination_folder_slot_2)

  print(f"File moved to: {os.path.join(destination_folder_slot_2, filename)}")
else:
  print(f"No data to export for {filename}")

No data to export for Koperasi List Delivery 14 March 2026 SLOT-2.xlsx


In [10]:
koperasi_13_df = df[df['driver_name'].str.contains('KOP-|kop-', case=True) & df['time_slot'].str.contains('slot-13|slot-sameday$', case=True)]

# Create an Excel writer object
filename = f"Koperasi List Delivery {date_str} SLOT-SAMEDAY.xlsx"
# Export the filtered DataFrame to Excel
if not koperasi_13_df.empty:
  formatted_df = koperasi_13_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  destination_folder_slot_13 = "/content/exported_files/slot_13"  # Replace with your desired folder path
  os.makedirs(destination_folder_slot_13, exist_ok=True)
  shutil.move(filename, destination_folder_slot_13)

  print(f"File moved to: {os.path.join(destination_folder_slot_13, filename)}")
else:
  print(f"No data to export for {filename}")

Koperasi List Delivery 14 March 2026 SLOT-SAMEDAY.xlsx is exported succesfully
File moved to: /content/exported_files/slot_13/Koperasi List Delivery 14 March 2026 SLOT-SAMEDAY.xlsx


In [11]:
# Extract data where driver_name has "DASH-" or "dash-" (case-insensitive)
dash_2_df = df[df['driver_name'].str.contains('DASH|dash', case=True) & df['time_slot'].str.contains('slot-2|slot-sameday03', case=True)]

# groups = dash_df.groupby('time_slot')

# Create an Excel writer object
filename = f"DASH List Delivery {date_str} SLOT-2.xlsx"
# Export the filtered DataFrame to Excel
if not dash_2_df.empty:
  formatted_df = dash_2_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  os.makedirs(destination_folder_slot_2, exist_ok=True)
  shutil.move(filename, destination_folder_slot_2)

  print(f"File moved to: {os.path.join(destination_folder_slot_2, filename)}")
else:
  print(f"No data to export for {filename}")

No data to export for DASH List Delivery 14 March 2026 SLOT-2.xlsx


In [12]:
# Extract data where driver_name has "DASH-" or "dash-" (case-insensitive)
dash_df = df[df['driver_name'].str.contains('DASH|dash', case=True)]
dash_2_df = dash_df[dash_df['time_slot'].str.contains('slot-2|slot-sameday03', case=True)]
dash_3_df = dash_2_df[dash_2_df['hubs'].str.contains('Pondok Kopi - JAKTIM|Bekasi - CIBUBUR', case=True)]

# groups = dash_df.groupby('time_slot')

# Create an Excel writer object
filename = f"[Koperasi] Support DASH List Delivery {date_str} SLOT-2.xlsx"
# Export the filtered DataFrame to Excel
if not dash_3_df.empty:
  formatted_df = dash_3_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  os.makedirs(destination_folder_slot_2, exist_ok=True)
  shutil.move(filename, destination_folder_slot_2)

  print(f"File moved to: {os.path.join(destination_folder_slot_2, filename)}")
else:
  print(f"No data to export for {filename}")

No data to export for [Koperasi] Support DASH List Delivery 14 March 2026 SLOT-2.xlsx


In [13]:
dash_13_df = df[df['driver_name'].str.contains('DASH|dash', case=True) & df['time_slot'].str.contains('slot-13|slot-sameday$', case=True)]

# groups = dash_df.groupby('time_slot')

# Create an Excel writer object
filename = f"DASH List Delivery {date_str} SLOT-SAMEDAY.xlsx"
# Export the filtered DataFrame to Excel
if not dash_13_df.empty:
  formatted_df = dash_13_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  destination_folder_slot_13 = "/content/exported_files/slot_13"  # Replace with your desired folder path
  os.makedirs(destination_folder_slot_13, exist_ok=True)
  shutil.move(filename, destination_folder_slot_13)

  print(f"File moved to: {os.path.join(destination_folder_slot_13, filename)}")
else:
  print(f"No data to export for {filename}")

DASH List Delivery 14 March 2026 SLOT-SAMEDAY.xlsx is exported succesfully
File moved to: /content/exported_files/slot_13/DASH List Delivery 14 March 2026 SLOT-SAMEDAY.xlsx


In [14]:
# Extract data where driver_name has "DASH-" or "dash-" (case-insensitive)
dash_df = df[df['driver_name'].str.contains('DASH|dash', case=True)]
dash_2_df = dash_df[dash_df['time_slot'].str.contains('slot-13|slot-sameday$', case=True)]
dash_3_df = dash_2_df[dash_2_df['hubs'].str.contains('Pondok Kopi - JAKTIM|Bekasi - CIBUBUR', case=True)]

# groups = borzo_df.groupby('time_slot')

# Create an Excel writer object
filename = f"[Koperasi] Support DASH List Delivery {date_str} SLOT-SAMEDAY.xlsx"
# Export the filtered DataFrame to Excel
if not dash_3_df.empty:
  formatted_df = dash_3_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  destination_folder_slot_13 = "/content/exported_files/slot_13"  # Replace with your desired folder path
  os.makedirs(destination_folder_slot_13, exist_ok=True)
  shutil.move(filename, destination_folder_slot_13)

  print(f"File moved to: {os.path.join(destination_folder_slot_13, filename)}")
else:
  print(f"No data to export for {filename}")

No data to export for [Koperasi] Support DASH List Delivery 14 March 2026 SLOT-SAMEDAY.xlsx


In [15]:
# Extract data where driver_name has "KJN-" or "kjn-" (case-insensitive)
kjn_2_df = df[df['driver_name'].str.contains('KJN-|kjn-', case=True) & df['time_slot'].str.contains('slot-2|slot-sameday03', case=True)]


# groups = kjn_df.groupby('time_slot')

# Create an Excel writer object
filename = f"KJN List Delivery {date_str} SLOT-2.xlsx"
# Export the filtered DataFrame to Excel
if not kjn_2_df.empty:
  formatted_df = kjn_2_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  os.makedirs(destination_folder_slot_2, exist_ok=True)
  shutil.move(filename, destination_folder_slot_2)

  print(f"File moved to: {os.path.join(destination_folder_slot_2, filename)}")
else:
  print(f"No data to export for {filename}")

No data to export for KJN List Delivery 14 March 2026 SLOT-2.xlsx


In [16]:
kjn_13_df = df[df['driver_name'].str.contains('KJN-|kjn-', case=True) & df['time_slot'].str.contains('slot-13|slot-sameday$', case=True)]

# groups = kjn_df.groupby('time_slot')

# Create an Excel writer object
filename = f"KJN List Delivery {date_str} SLOT-SAMEDAY.xlsx"
# Export the filtered DataFrame to Excel
if not kjn_13_df.empty:
  formatted_df = kjn_13_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  destination_folder_slot_13 = "/content/exported_files/slot_13"  # Replace with your desired folder path
  os.makedirs(destination_folder_slot_13, exist_ok=True)
  shutil.move(filename, destination_folder_slot_13)

  print(f"File moved to: {os.path.join(destination_folder_slot_13, filename)}")
else:
  print(f"No data to export for {filename}")

No data to export for KJN List Delivery 14 March 2026 SLOT-SAMEDAY.xlsx


In [17]:
# # Extract data where driver_name has "GOGO-" or "gogo-" (case=True)
# gogo_2_df = df[df['driver_name'].str.contains('GOGO-|gogo-', case=True) & df['time_slot'].str.contains('slot-2|slot-sameday03', case=True)]


# # groups = kjn_df.groupby('time_slot')

# # Create an Excel writer object
# filename = f"GO N GO List Delivery {date_str} SLOT-2.xlsx"
# # Export the filtered DataFrame to Excel
# if not gogo_2_df.empty:
#   formatted_df = gogo_2_df.style \
#     .apply(cash_on_delivery_payment, axis=1) \
#     .set_table_styles([
#         {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
#     ])

#   formatted_df.to_excel(filename, index=False)

#   print(f"{filename} is exported succesfully")

#   os.makedirs(destination_folder_slot_2, exist_ok=True)
#   shutil.move(filename, destination_folder_slot_2)

#   print(f"File moved to: {os.path.join(destination_folder_slot_2, filename)}")
# else:
#   print(f"No data to export for {filename}")

In [18]:
# gogo_13_df = df[df['driver_name'].str.contains('GOGO-|gogo-', case=True) & df['time_slot'].str.contains('slot-13|slot-sameday$', case=True)]

# # groups = gogo_df.groupby('time_slot')

# # Create an Excel writer object
# filename = f"GO N GO List Delivery {date_str} SLOT-SAMEDAY.xlsx"
# # Export the filtered DataFrame to Excel
# if not gogo_13_df.empty:
#   formatted_df = gogo_13_df.style \
#     .apply(cash_on_delivery_payment, axis=1) \
#     .set_table_styles([
#         {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
#     ])

#   formatted_df.to_excel(filename, index=False)

#   print(f"{filename} is exported succesfully")

#   destination_folder_slot_13 = "/content/exported_files/slot_13"  # Replace with your desired folder path
#   os.makedirs(destination_folder_slot_13, exist_ok=True)
#   shutil.move(filename, destination_folder_slot_13)

#   print(f"File moved to: {os.path.join(destination_folder_slot_13, filename)}")
# else:
#   print(f"No data to export for {filename}")

In [19]:
# Extract data where driver_name has "BORZO" or "borzo" (case-insensitive)
borzo_2_df = df[df['driver_name'].str.contains('BORZO|borzo', case=True) & df['time_slot'].str.contains('slot-2|slot-sameday03', case=True)]

# groups = kjn_df.groupby('time_slot')

# Create an Excel writer object
filename = f"Borzo List Delivery {date_str} SLOT-2.xlsx"
# Export the filtered DataFrame to Excel
if not borzo_2_df.empty:
  formatted_df = borzo_2_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  os.makedirs(destination_folder_slot_2, exist_ok=True)
  shutil.move(filename, destination_folder_slot_2)

  print(f"File moved to: {os.path.join(destination_folder_slot_2, filename)}")
else:
  print(f"No data to export for {filename}")

No data to export for Borzo List Delivery 14 March 2026 SLOT-2.xlsx


In [20]:
borzo_13_df = df[df['driver_name'].str.contains('BORZO|borzo', case=True) & df['time_slot'].str.contains('slot-13|slot-sameday$', case=True)]

# groups = borzo_df.groupby('time_slot')

# Create an Excel writer object
filename = f"Borzo List Delivery {date_str} SLOT-SAMEDAY.xlsx"
# Export the filtered DataFrame to Excel
if not borzo_13_df.empty:
  formatted_df = borzo_13_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  destination_folder_slot_13 = "/content/exported_files/slot_13"  # Replace with your desired folder path
  os.makedirs(destination_folder_slot_13, exist_ok=True)
  shutil.move(filename, destination_folder_slot_13)

  print(f"File moved to: {os.path.join(destination_folder_slot_13, filename)}")
else:
  print(f"No data to export for {filename}")

No data to export for Borzo List Delivery 14 March 2026 SLOT-SAMEDAY.xlsx


In [21]:
# Extract data where driver_name has "BORZO" or "borzo" (case-insensitive)
borzo_df = df[df['driver_name'].str.contains('BORZO|borzo', case=True)]
borzo_2_df = borzo_df[borzo_df['time_slot'].str.contains('slot-2|slot-sameday03', case=True)]
borzo_3_df = borzo_2_df[borzo_2_df['hubs'].str.contains('Pondok Kopi - JAKTIM', case=True)]

# groups = dash_df.groupby('time_slot')

# Create an Excel writer object
filename = f"[Koperasi] Support Borzo List Delivery {date_str} SLOT-2.xlsx"
# Export the filtered DataFrame to Excel
if not borzo_3_df.empty:
  formatted_df = borzo_3_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  os.makedirs(destination_folder_slot_2, exist_ok=True)
  shutil.move(filename, destination_folder_slot_2)

  print(f"File moved to: {os.path.join(destination_folder_slot_2, filename)}")
else:
  print(f"No data to export for {filename}")

No data to export for [Koperasi] Support Borzo List Delivery 14 March 2026 SLOT-2.xlsx


In [22]:
# Extract data where driver_name has "BORZO" or "borzo" (case-insensitive)
borzo_df = df[df['driver_name'].str.contains('BORZO|borzo', case=True)]
borzo_2_df = borzo_df[borzo_df['time_slot'].str.contains('slot-13|slot-sameday$', case=True)]
borzo_3_df = borzo_2_df[borzo_2_df['hubs'].str.contains('Pondok Kopi - JAKTIM', case=True)]

# groups = borzo_df.groupby('time_slot')

# Create an Excel writer object
filename = f"[Koperasi] Support Borzo List Delivery {date_str} SLOT-SAMEDAY.xlsx"
# Export the filtered DataFrame to Excel
if not borzo_3_df.empty:
  formatted_df = borzo_3_df.style \
    .apply(cash_on_delivery_payment, axis=1) \
    .set_table_styles([
      {'selector': 'th', 'props': [('border', '0.5px solid #D3D3D3')]}  # Add border to header cells
  ])

  formatted_df.to_excel(filename, index=False)

  print(f"{filename} is exported succesfully")

  destination_folder_slot_13 = "/content/exported_files/slot_13"  # Replace with your desired folder path
  os.makedirs(destination_folder_slot_13, exist_ok=True)
  shutil.move(filename, destination_folder_slot_13)

  print(f"File moved to: {os.path.join(destination_folder_slot_13, filename)}")
else:
  print(f"No data to export for {filename}")

No data to export for [Koperasi] Support Borzo List Delivery 14 March 2026 SLOT-SAMEDAY.xlsx


In [23]:
def zip_exported_files(folder_path, zip_filename):
  """Zips all files within a folder if the folder is not empty.

  Args:
      folder_path (str): Path to the folder containing the files.
      zip_filename (str): Name of the output zip file.
  """
  if os.path.exists(folder_path) and os.listdir(folder_path):
    with ZipFile(zip_filename, 'w') as zipf:
      for root, _, files_list in os.walk(folder_path):
        for file in files_list:
          file_path = os.path.join(root, file)
          zipf.write(file_path, os.path.relpath(file_path, folder_path))

    print(f"Folder '{folder_path}' zipped to '{zip_filename}'")
    files.download(zip_filename)
  else:
    print(f"Folder '{folder_path}' is empty. No zip file created.")


# Example usage:
destination_folder_slot_2 = "/content/exported_files/slot_2"  # Replace with your folder path
zip_filename_slot_2 = "slot-2.zip"  # Replace with your desired zip filename

zip_exported_files(destination_folder_slot_2, zip_filename_slot_2)


# Example usage:
destination_folder_slot_13 = "/content/exported_files/slot_13"  # Replace with your folder path
zip_filename_slot_13 = "slot-13.zip"  # Replace with your desired zip filename

zip_exported_files(destination_folder_slot_13, zip_filename_slot_13)

Folder '/content/exported_files/slot_2' is empty. No zip file created.
Folder '/content/exported_files/slot_13' zipped to 'slot-13.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>